# Requête HTTP 

Un requête HTTP est une requête basée sur le protocole TCP, elle fait partie de la couche application de la couche OSI. Elle permet d'accéder aux données mise à disposition sur une adresse IP (ou url résolue par un DNS) et un port. 

Les deux ports les plus utilisés dans le web sont le 80 pour les sites en HTTP et le 443 pour les sites en HTTPS. HTTPS est une variable du protocole HTTP basé sur le protocole TLS.

Il existe de nombreux types de requêtes selon la convention `REST`: 
- GET
- POST
- PUT 
- DELETE
- UPDATE.

Dans notre cas, nous allons utiliser la plupart du temps des GET et potentiellement des POST. 
- Le GET permet comme son nom l'indique de récupérer des informations en fonction de certains paramètres. 
- Le POST nécessite un envoi de données pour récupérer des données. Le body du post est, la plupart du temps, envoyé sous la forme d'un objet JSON.

Ces requêtes encapsulent un certain nombre de paramètres qui permettent soient d'identifier une provenance et un utilisateur ou de réaliser différentes actions.

In [1]:
import requests

In [2]:
url = "https://www.esiee.fr/"
response = requests.get(url)
response.status_code

200

Il existe deux méthodes pour récupérer le contenu de la page :

- `response.text` qui permet de retourner le texte sous la forme d'une chaine de charactères.
- `response.content` qui permet de récupérer le contenu de la page sous la forme de bytes

In [3]:
type(response.content)

bytes

In [4]:
type(response.text)

str

Pour récupérer les 1000 premiers charactères de la page :

In [5]:
response.text[0:1000]

'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;école de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande école d&#039;ingénieur dans les domaines des transitions numérique, énergétique et environnementale. Classée parmi les meilleures écoles d&#039;ingénieur selon le classement de l&#039;Etudiant. Habilitée par la Commission des Titres d&#039;Ingénieur (CTI). Membre de la Conférence des Grandes Ecoles (CGE). " />\n<meta name="viewport" content="width=device-w

Pour récupérer les headers HTTP de la réponse :

In [6]:
response.headers

{'Date': 'Tue, 17 Nov 2026 15:30:25 GMT', 'Server': 'Apache', 'Content-Language': 'fr', 'Vary': 'Accept-Encoding', 'Content-Encoding': 'gzip', 'X-UA-Compatible': 'IE=edge', 'X-Content-Type-Options': 'nosniff', 'Content-Length': '17417', 'Content-Type': 'text/html; charset=utf-8', 'X-Varnish': '52365151 55739557', 'Age': '127', 'Via': '1.1 varnish (Varnish/7.1)', 'Accept-Ranges': 'bytes', 'Connection': 'keep-alive'}

On peut modifier les paramètres de la requête et/ou ses headers. On peut par exemple ajouter un UserAgent (identifiant de l'initiateur de la requête) et un timeout de 10 secondes :

In [7]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
response = requests.get(url, headers=headers, timeout = 10)
response.content[0:1000]

b'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;\xc3\xa9cole de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande \xc3\xa9cole d&#039;ing\xc3\xa9nieur dans les domaines des transitions num\xc3\xa9rique, \xc3\xa9nerg\xc3\xa9tique et environnementale. Class\xc3\xa9e parmi les meilleures \xc3\xa9coles d&#039;ing\xc3\xa9nieur selon le classement de l&#039;Etudiant. Habilit\xc3\xa9e par la Commission des Titres d&#039;Ing\xc3\xa9nieur (CTI). Membre de la Conf\xc3\x

## Exercice

## Exercice 1

- Créer une classe Python permettant de faire des requêtes HTTP.
- Cette classe doit utiliser toujours le même UserAgent.
- Le TimeOut sera spécifié à chaque appelle avec une valeur par défaut.
- Un mécanisme de retry sera mis en place de façon recursive.

## Exercice 2

- Faire une fonction permettant de supprimer tous les espaces supperflus d'une string
- Faire une fonction qui prend une string html et renvois une string intelligible (enlever les caractères spéciaux,
- Récupérer le domaine en fonction d'un url

### Solution Exercice 1

Classe `HttpClient` :
- **User-Agent fixe** : stocké en attribut de classe, envoyé dans les headers à chaque requête.
- **Timeout** : paramètre de `get()` avec une valeur par défaut (10 s), modifiable à chaque appel.
- **Retry récursif** : en cas d'échec, `get()` se rappelle elle-même avec un compteur `retries` diminué de 1, jusqu'à 0 (alors on relance l'erreur).

In [8]:
import time
import requests


class HttpClient:
    """Client HTTP simple : User-Agent fixe, timeout par défaut, retry récursif."""

    USER_AGENT = (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )

    def __init__(self, default_timeout=10, max_retries=3, wait=1):
        self.default_timeout = default_timeout  # timeout par défaut (secondes)
        self.max_retries = max_retries          # nombre de nouvelles tentatives
        self.wait = wait                        # pause entre deux tentatives (secondes)
        self.headers = {"User-Agent": self.USER_AGENT}

    def get(self, url, timeout=None, retries=None):
        """Fait un GET sur `url`. Réessaie récursivement en cas d'échec."""
        if timeout is None:
            timeout = self.default_timeout
        if retries is None:
            retries = self.max_retries

        try:
            response = requests.get(url, headers=self.headers, timeout=timeout)
            response.raise_for_status()  # lève une erreur si code 4xx / 5xx
            return response
        except requests.RequestException as error:
            if retries <= 0:
                print(f"Abandon pour {url} : {error}")
                raise
            print(f"Erreur ({type(error).__name__}), nouvel essai... ({retries} restant(s))")
            time.sleep(self.wait)
            return self.get(url, timeout=timeout, retries=retries - 1)  # appel récursif


Test 1 : une requête qui fonctionne (timeout par défaut, puis timeout personnalisé).

In [9]:
client = HttpClient()

response = client.get("https://www.esiee.fr/")
print(response.status_code)
print(response.request.headers["User-Agent"])  # on vérifie le User-Agent envoyé

response = client.get("https://www.esiee.fr/", timeout=5)
print(response.status_code)


200
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36
200


Test 2 : une URL qui n'existe pas, pour voir le mécanisme de retry récursif.

In [10]:
try:
    client.get("https://ce-site-nexiste-pas-123456.fr", timeout=3, retries=2)
except requests.RequestException:
    print("Échec après tous les essais, comme prévu.")


Erreur (ConnectionError), nouvel essai... (2 restant(s))


Erreur (ConnectionError), nouvel essai... (1 restant(s))


Abandon pour https://ce-site-nexiste-pas-123456.fr : HTTPSConnectionPool(host='ce-site-nexiste-pas-123456.fr', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xffff8470aed0>: Failed to resolve 'ce-site-nexiste-pas-123456.fr' ([Errno -2] Name or service not known)"))
Échec après tous les essais, comme prévu.


### Solution Exercice 2

Trois fonctions utilitaires, réutilisées dans les exercices suivants :
- `clean_spaces` : remplace toute suite d'espaces / tabulations / retours à la ligne par un seul espace.
- `html_to_text` : enlève les balises HTML (et le contenu des `<script>` / `<style>`), décode les entités (`&amp;`, `&nbsp;`...), supprime les caractères spéciaux et les espaces superflus.
- `get_domain` : extrait le domaine d'une URL avec `urllib.parse`.

In [11]:
import re
import html
import unicodedata
from urllib.parse import urlparse

from bs4 import BeautifulSoup


def clean_spaces(text):
    """Supprime les espaces superflus (début, fin, et espaces multiples)."""
    return re.sub(r"\s+", " ", text).strip()


def html_to_text(html_string):
    """Transforme une chaîne HTML en texte lisible."""
    soup = BeautifulSoup(html_string, "html.parser")
    for tag in soup(["script", "style", "noscript"]):  # contenu non lisible
        tag.decompose()
    text = soup.get_text(" ")
    text = html.unescape(text)                     # &amp; -> &, &eacute; -> é ...
    text = unicodedata.normalize("NFKC", text)     # \xa0 (espace insécable) -> espace
    text = re.sub(r"[^\w\s.,;:!?'’()€%&-]", " ", text)  # caractères spéciaux
    return clean_spaces(text)


def get_domain(url, keep_www=False):
    """Renvoie le domaine d'une URL (sans le 'www.' par défaut)."""
    domain = (urlparse(url).hostname or "").lower()
    if not keep_www and domain.startswith("www."):
        domain = domain[4:]
    return domain


In [12]:
print(repr(clean_spaces("   Bonjour     le \n\n  monde \t !  ")))

exemple_html = """<html><head><style>p {color: red;}</style></head>
<body><h1>Titre &amp; sous-titre</h1>
<p>Un   texte avec des <b>balises</b> &nbsp; et des ★ symboles ♥ !</p>
<script>alert("coucou")</script></body></html>"""
print(repr(html_to_text(exemple_html)))

print(get_domain("https://www.esiee.fr/fr/formations?page=2"))
print(get_domain("https://www.esiee.fr/fr/formations", keep_www=True))
print(get_domain("http://blog.example.com:8080/article"))


'Bonjour le monde !'
'Titre & sous-titre Un texte avec des balises et des symboles !'
esiee.fr
www.esiee.fr
blog.example.com


# Exploitation du HTML  

Ici, il faut récupérer le code HTML d'un site web à partir d'une requête. Lorsque vous avez récupéré le texte d'un site il faut le parser. Pour cela, on utilise BeautifulSoup qui permet de transformer la structure HTML en objet Python. Cela permet de récupérer efficacement les données qui nous intéresse.

Pour les webmasters, le blocage le plus souvent mis en place et un blocage sur le User-Agent. Le User-Agent est un paramètre intégré dans la requête HTTP réalisé par le Navigateur pour envoyer au front des informations basiques :

- la version du Navigateur,
- la version de l'OS
- Le type de gestionnaire graphique (Gecko)
- le type de device utilisé

Exemple de User Agent :  

`Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0`

Commençons à utiliser `BeautifulSoup`, il est normalement déjà installé, le cas échéant executez les lignes suivantes : 

In [13]:
import requests
from bs4 import BeautifulSoup

Pour transformer une requête (requests) en objet BeautifulSoup :

In [14]:
response = requests.get(url)
soup = BeautifulSoup(response.text)

Pour trouver tous les liens d'une page, on récupère la balise `a` qui permet de gérer les liens en HTML :

In [15]:
soup.find_all("a")[0:10]

[<a href="/#content">Aller au contenu</a>,
 <a href="/#menu">Aller au menu</a>,
 <a href="/plan-du-site/">Plan du site</a>,
 <a href="/actualites/journees-portes-ouvertes-2026-2027" target="_blank" title="Ouvre une nouvelle fenêtre">Portes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h </a>,
 <a href="/"><img alt="ESIEE PARIS" class="a42-ac-replace-img" src="/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/logo-esiee.svg"/></a>,
 <a href="/brochures-1">Brochures</a>,
 <a href="/informations/etudiantes-et-etudiants">Espace élèves</a>,
 <a href="/" hreflang="fr-FR" title="Français">
 <span>Fr</span>
 </a>,
 <a href="/en/" hreflang="en-US" title="English">
 <span>En</span>
 </a>,
 <a href="/candidater-1">Candidater</a>]

On peut aussi préciser la classe HTML qu'on veut récupérer :

```python
soup.find_all(class_="<CLASS_NAME>")[0:10]
```

Ici par exemple: 

In [16]:
soup.find_all(class_="toggler")[0:5]

[<button aria-controls="searchbox-header-form" aria-expanded="false" class="toggler">
 <i class="fa-solid fa-magnifying-glass"></i>
 <i class="fa-solid fa-xmark"></i>
 <span class="sr-only">
 <span class="display">Afficher</span><span class="hide">Masquer</span> la recherche
 		</span>
 </button>,
 <button aria-controls="submenu-40" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>L'école</button>,
 <button aria-controls="submenu-563" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Gouvernance et conseils</button>,
 <button aria-controls="submenu-65" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Départements d'enseignements et de recherche</button>,
 <button aria-controls="su

Pour récupérer le text sans les balises HTML :

In [17]:
soup.text[0:1000]

"\n\n\n\nESIEE Paris, l'école de l'innovation technologique | ESIEE Paris\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAller au contenu\nAller au menu\nPlan du site\n\n\n\n\n\n\n\nPortes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h \n\n\n\n\n\nMasquer l'alerte\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBrochuresEspace élèves\n\n\n\nFr\n\n\n\n\nEn\n\n\n\n\n\n\n\n\n\nAfficherMasquer la recherche\r\n\t\t\n\n\n\nSaisissez votre recherche\xa0:\n\nLancer la recherche\n\n\n\nCandidater\n\nAfficherMasquer le menu\n\n\n\n\n\nRetour au menu principalAfficherMasquer le sous menu\xa0: L'écolePourquoi choisir ESIEE Paris ?AfficherMasquer le sous menu\xa0: Gouvernance et conseilsGouvernance et conseilsConseil scientifiqueAfficherMasquer le sous menu\xa0: Départements d'enseignements et de rechercheInformatique et télécommunicationsIngénierie des systèmes cyberphysiquesIngénierie industrielleSanté, énergie et environnement durableManagement, sciences humaines et languesCorps professoral

## Exercice
### Exercice 3

Améliorer la classe développé précédemment.

- Ajouter une méthode pour récupérer l'objet soup d'un url
- Récupérer une liste de User Agent et effectuer une rotation aléatoire sur celui à utiliser
- Utiliser cette classe pour parser une page HTML et récupérer : le titre, tous les H1 (si ils existent), les liens vers les images, les liens sortants vers d'autres sites, et le texte principal.

### Solution Exercice 3

On améliore la classe de l'exercice 1 en créant `Scraper`, qui **hérite** de `HttpClient` :
- **Rotation aléatoire du User-Agent** : à chaque requête (et donc aussi à chaque nouvel essai du retry), un User-Agent est tiré au hasard dans une liste.
- `get_soup(url)` : renvoie directement l'objet `BeautifulSoup` de la page.
- `parse_page(url)` : renvoie un dictionnaire avec le titre, les H1, les liens des images, les liens sortants et le texte principal (en réutilisant les fonctions de l'exercice 2).

In [18]:
import random
from urllib.parse import urljoin


class Scraper(HttpClient):
    """HttpClient + rotation de User-Agent + outils BeautifulSoup."""

    USER_AGENTS = [
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    ]

    def get(self, url, timeout=None, retries=None):
        # Nouveau User-Agent tiré au hasard avant chaque requête
        self.headers = {"User-Agent": random.choice(self.USER_AGENTS)}
        return super().get(url, timeout=timeout, retries=retries)

    def get_soup(self, url, timeout=None):
        """Renvoie l'objet BeautifulSoup de la page."""
        response = self.get(url, timeout=timeout)
        return BeautifulSoup(response.text, "html.parser")

    def parse_page(self, url):
        """Extrait les informations principales d'une page HTML."""
        soup = self.get_soup(url)
        domain = get_domain(url)

        # Titre de la page
        title = clean_spaces(soup.title.get_text()) if soup.title else None

        # Tous les H1 (si le H1 ne contient qu'une image, on prend son texte alternatif)
        h1 = []
        for h in soup.find_all("h1"):
            text = clean_spaces(h.get_text())
            if not text and h.find("img", alt=True):
                text = clean_spaces(h.find("img")["alt"])
            if text:
                h1.append(text)

        # Liens des images (URL absolues). Les sites utilisent souvent le "lazy loading" :
        # la vraie image est dans data-src et src contient une image vide en "data:..."
        images = []
        for img in soup.find_all("img"):
            src = img.get("data-src") or img.get("src")
            if src and not src.startswith("data:"):
                link = urljoin(url, src)
                if link not in images:
                    images.append(link)

        # Liens sortants : liens http(s) vers un autre domaine
        external_links = set()
        for a in soup.find_all("a", href=True):
            link = urljoin(url, a["href"])
            if link.startswith("http") and get_domain(link) != domain:
                external_links.add(link)

        # Texte principal : <main> ou <article> s'ils existent, sinon <body>
        main = soup.find("main") or soup.find("article") or soup.body or soup
        main_text = html_to_text(str(main))

        return {
            "url": url,
            "domain": domain,
            "title": title,
            "h1": h1,
            "images": images,
            "external_links": sorted(external_links),
            "main_text": main_text,
        }


In [19]:
scraper = Scraper()

# La rotation du User-Agent : on fait 3 requêtes et on regarde celui envoyé
for _ in range(3):
    r = scraper.get("https://www.esiee.fr/")
    print(r.request.headers["User-Agent"][:60], "...")


Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4) AppleWebKit/605 ...
Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4) AppleWebKit/605 ...


Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/ ...


In [20]:
page = scraper.parse_page("https://www.esiee.fr/")

print("Titre :", page["title"])
print("H1 :", page["h1"])
print(f"{len(page['images'])} images, ex :", page["images"][:3])
print(f"{len(page['external_links'])} liens sortants, ex :", page["external_links"][:5])
print("Texte principal :", page["main_text"][:500], "...")


Titre : ESIEE Paris, l'école de l'innovation technologique | ESIEE Paris
H1 : ['ESIEE PARIS']
43 images, ex : ['https://www.esiee.fr/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/logo-esiee.svg', 'https://www.esiee.fr/fileadmin/user_upload/Fichiers/image-home/HomeESIEEParis2026-1920x1080.webp', 'https://www.esiee.fr/fileadmin/_processed_/9/5/csm_actu-jpo-1344x840_6889669208.jpg']
12 liens sortants, ex : ['http://esiee.jobteaser.com/fr/recruiter_account/job_offers', 'http://www.youtube.com/esieeparis', 'https://bsky.app/profile/esiee.fr', 'https://www.cci-paris-idf.fr/fr/education', 'https://www.cge.asso.fr/']
Texte principal : ESIEE Paris, l’école de l'innovation technologique ESIEE Paris, 1re école d’ingénieurs post-bac selon le dernier classement de l'Usine Nouvelle 2026. Classement L'Etudiant 2026 ESIEE Paris, n 3 des écoles d’ingénieurs post-bac d’Ile-de-France selon le Palmarès L’Etudiant 2026 . 8 mars 2026 Journée internationale des droits des femmes ESIEE Paris s'eng

# Exploitation des appels d'API



Losque le front du site récupère des données sur une API gérée par le back, un appel d'API est réalisé. Cet appel est recensé dans les appels réseaux. Il est alors possible de re-jouer cet appel pour récupérer à nouveau les données. Il est très facile de récupérer ces appels dans l'onglet Network de la console développeur de Chrome ou FireFox. La console vous permet de copier le code CURL de la requête et vous pouvez ensuite la transformer en code Python depuis le site https://curl.trillworks.com/.

Souvent les APIs sont bloquées avec certains paramètres. L'API vérifie que dans les headers de la requête HTTP ces paramètres sont présents :
* un token généré à la volée avec des protocoles OAuth2 (ou moins développés).
* un referer provenant du site web (la source de la requête), très facile à falsifier.



## Exercice 
### Exercice 4

- Utiliser les informations développées plus haut pour récupérer les premiers résultats d'une recherche d'une requête
sur Google. 

Tips : 

- Ouvrir les outils de développements de Chrome ou Firefox
- Onglet Network
- Fouiller dans les requêtes pour voir à quoi ressemble un appel API Google
- Utilisez beautiful soup pour convertir le contenu de la request en objet et accéder aux balises

### Solution Exercice 4

**Démarche (outils de développement) :** sur google.fr, ouvrir les outils de développement (`Cmd + Option + I`), onglet **Network**, lancer une recherche. La requête principale est un simple **GET** sur `https://www.google.com/search` avec les paramètres `q` (la recherche), `hl` (la langue) et `num` (le nombre de résultats). On la rejoue avec `requests`, puis on parse le HTML avec BeautifulSoup : chaque résultat est un lien `<a>` qui contient un titre `<h3>`.

**Limite :** Google bloque de plus en plus les scripts (page de consentement, captcha, page « activez JavaScript »). La fonction détecte ce cas et renvoie une liste vide. Pour avoir des résultats exploitables, on applique exactement la même méthode à **DuckDuckGo**, dont la version HTML (`html.duckduckgo.com`) fonctionne sans JavaScript.

In [21]:
from urllib.parse import parse_qs


def google_search(query, n=10, scraper=scraper):
    """Rejoue l'appel GET de Google et renvoie [{'title', 'url'}]."""
    url = "https://www.google.com/search"
    params = {"q": query, "hl": "fr", "num": n}
    response = requests.get(url, params=params, headers={"User-Agent": random.choice(scraper.USER_AGENTS)},
                            cookies={"CONSENT": "YES+"}, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    results = []
    for h3 in soup.find_all("h3"):
        a = h3.find_parent("a", href=True)
        if a is None:
            continue
        link = a["href"]
        if link.startswith("/url?"):  # ancien format : /url?q=<vraie_url>&...
            link = parse_qs(urlparse(link).query).get("q", [link])[0]
        if link.startswith("http"):
            results.append({"title": clean_spaces(h3.get_text()), "url": link})

    if not results:
        print(f"Google a bloqué la requête ou demande JavaScript (code {response.status_code}) : aucun résultat récupéré.")
    return results[:n]


def duckduckgo_search(query, n=10, scraper=scraper):
    """Même méthode sur la version HTML de DuckDuckGo.

    Dans l'onglet Network, on voit que le formulaire de html.duckduckgo.com envoie
    un POST avec le paramètre q : on rejoue exactement cet appel.
    """
    url = "https://html.duckduckgo.com/html/"
    headers = {
        "User-Agent": random.choice(scraper.USER_AGENTS),
        "Referer": "https://html.duckduckgo.com/",
        "Accept-Language": "fr-FR,fr;q=0.9",
    }
    response = requests.post(url, data={"q": query, "kl": "fr-fr"}, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    results = []
    for a in soup.select("a.result__a"):
        link = a["href"]
        if "uddg=" in link:  # lien de redirection : //duckduckgo.com/l/?uddg=<vraie_url>
            link = parse_qs(urlparse(link).query)["uddg"][0]
        results.append({"title": clean_spaces(a.get_text()), "url": link, "domain": get_domain(link)})

    if not results:
        # Diagnostic : code HTTP + début du texte de la page reçue
        print(f"DuckDuckGo : aucun résultat (code {response.status_code}).")
        print("Début de la page reçue :", html_to_text(response.text)[:300])
    return results[:n]


In [22]:
requete = "ESIEE Paris data science"

print("=== Google ===")
for i, res in enumerate(google_search(requete), 1):
    print(i, res["title"], "->", res["url"])

print("\n=== DuckDuckGo ===")
for i, res in enumerate(duckduckgo_search(requete), 1):
    print(i, res["title"], "->", res["url"])


=== Google ===


Google a bloqué la requête ou demande JavaScript (code 200) : aucun résultat récupéré.

=== DuckDuckGo ===


DuckDuckGo : aucun résultat (code 202).
Début de la page reçue : DuckDuckGo DuckDuckGo Unfortunately, bots use DuckDuckGo too. Please complete the following challenge to confirm this search was made by a human. Select all squares containing a duck: Submit Images not loading? Please email the following code to: error-lite c0d8 duckduckgo.com Code: c0d8


# Exercice Final  

Exercice Final
Utilisez tout ce que vous avez appris pour récupérer des articles de News avec une catégorie. Il est souvent intéressant de partir des flux RSS pour commencer :

Les données doivent comprendre :
- Le texte important propre
- L'url
- Le domaine
- la catégorie
- Le titre de l'article
- Le titre de la page
- (Facultatif) : les images

Tips : 

- Taper le nom de votre média favoris + RSS (par exemple : https://www.lemonde.fr/rss/)
- Aller dans le DOM de la page 
- Trouver les catégories et les liens vers les articles

### Solution Exercice Final

**Principe :** un flux RSS est la liste des derniers articles d'un média, dans un format XML standard (`<item>` avec `<title>`, `<link>`, `<pubDate>`, `<description>`...). Les médias ont un flux **par rubrique** : la catégorie d'un article est donc simplement le nom du flux d'où il vient.

**Étapes :**
1. Lire chaque flux RSS (BeautifulSoup avec le parser `"xml"`) : on obtient la liste des articles (titre, lien, date, résumé, image).
2. Visiter chaque article avec la classe `Scraper` de l'exercice 3 et en extraire : le titre de la page, le titre de l'article (`<h1>`), le texte principal propre, les images.
3. Ajouter l'URL, le domaine (`get_domain` de l'exercice 2) et la catégorie.
4. Tout ranger dans un `DataFrame` pandas.

**Choix des médias :** les flux du Monde sont lisibles, mais ses pages d'articles renvoient « Votre trafic a été identifié comme automatisé (bot) » (code 402). On respecte ce blocage et on utilise **franceinfo** et **20 Minutes**, dont les articles sont accessibles.

**Extraction du texte principal :** chaque site range ses articles différemment. On utilise une méthode générique : on cherche le bloc HTML qui contient **le plus de texte en paragraphes `<p>`** (c'est presque toujours le corps de l'article), puis on enlève les phrases parasites (newsletter, cookies...). Si l'article n'a presque pas de texte (vidéo, direct), on garde le résumé du flux RSS.

In [23]:
import time
from collections import defaultdict
from urllib.parse import urlunparse

import pandas as pd

# Flux RSS par média et par catégorie
SOURCES = {
    "franceinfo": {
        "Politique": "https://www.francetvinfo.fr/politique.rss",
        "Économie": "https://www.francetvinfo.fr/economie.rss",
        "Sport": "https://www.francetvinfo.fr/sports.rss",
        "Culture": "https://www.francetvinfo.fr/culture.rss",
        "Sciences": "https://www.francetvinfo.fr/sciences.rss",
    },
    "20 Minutes": {
        "Économie": "https://www.20minutes.fr/feeds/rss-economie.xml",
        "Sport": "https://www.20minutes.fr/feeds/rss-sport.xml",
        "Culture": "https://www.20minutes.fr/feeds/rss-culture.xml",
        "Sciences": "https://www.20minutes.fr/feeds/rss-sciences.xml",
    },
}

N_PER_FEED = 3   # nombre d'articles récupérés par flux (on reste raisonnable)
PAUSE = 1        # secondes entre deux requêtes, pour ne pas surcharger les sites

# Phrases parasites à ne pas garder dans le texte d'un article
PARASITES = re.compile(
    r"newsletter|cookies|consentement|connectez-vous|abonnez|France Télévisions utilise"
    r"|Ce texte correspond à une partie|Activez ou désactivez|Lire aussi|Publié le|Mis à jour",
    re.IGNORECASE,
)


In [24]:
def clean_url(url):
    """Enlève les paramètres de suivi (?at_medium=..., #xtor=...) d'une URL."""
    p = urlparse(url.strip())
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))


def read_feed(feed_url, scraper=scraper):
    """Lit un flux RSS et renvoie la liste des articles qu'il contient."""
    response = scraper.get(feed_url)
    feed = BeautifulSoup(response.content, "xml")  # parser XML (et non HTML)

    entries = []
    for item in feed.find_all("item"):
        # Image fournie par le flux : <enclosure url=...> ou <media:content url=...>
        desc = item.description.get_text() if item.description else ""
        image_tag = item.find("enclosure") or item.find("content", url=True)
        entries.append({
            "rss_title": clean_spaces(item.title.get_text()) if item.title else None,
            "url": clean_url(item.link.get_text()),
            "published": item.pubDate.get_text() if item.pubDate else None,
            # Le résumé contient parfois du HTML : on le nettoie seulement dans ce cas
            "summary": (html_to_text(desc) if "<" in desc else clean_spaces(html.unescape(desc))),
            "rss_image": image_tag.get("url") if image_tag else None,
        })
    return entries


def extract_main_text(soup):
    """Texte principal : le bloc qui contient le plus de texte en paragraphes <p>."""
    for tag in soup(["script", "style", "noscript", "aside", "nav", "footer", "header", "form"]):
        tag.decompose()

    score = defaultdict(int)  # bloc parent -> nombre de caractères de ses paragraphes
    for p in soup.find_all("p"):
        text = clean_spaces(p.get_text(" "))
        if len(text) >= 40 and not PARASITES.search(text):
            score[p.parent] += len(text)
    if not score:
        return ""

    best_block = max(score, key=score.get)
    paragraphs = []
    for p in best_block.find_all("p", recursive=False):
        text = clean_spaces(p.get_text(" "))
        if len(text) >= 40 and not PARASITES.search(text):
            paragraphs.append(text)
    return "\n".join(paragraphs)


def scrape_article(entry, category, media, scraper=scraper):
    """Visite un article et renvoie toutes les informations demandées."""
    soup = scraper.get_soup(entry["url"])

    page_title = clean_spaces(soup.title.get_text()) if soup.title else None
    h1 = soup.find("h1")
    article_title = clean_spaces(h1.get_text(" ")) if h1 else entry["rss_title"]

    # Images : image de partage (og:image), image du flux, puis images de l'article
    images = []
    og = soup.find("meta", property="og:image")
    for link in [og.get("content") if og else None, entry["rss_image"]]:
        if link and link not in images:
            images.append(link)
    article_block = soup.find("article") or soup.find("main")
    if article_block:
        for img in article_block.find_all("img"):
            src = img.get("data-src") or img.get("src")
            if src and not src.startswith("data:"):
                link = urljoin(entry["url"], src)
                if link not in images:
                    images.append(link)

    text = extract_main_text(soup)
    text_source = "article"
    if len(text) < 300:  # vidéo, direct... : on garde le résumé du flux
        text, text_source = entry["summary"], "résumé RSS"

    return {
        "media": media,
        "category": category,
        "article_title": article_title,
        "page_title": page_title,
        "url": entry["url"],
        "domain": get_domain(entry["url"]),
        "published": entry["published"],
        "text": text,
        "text_source": text_source,
        "text_length": len(text),
        "images": images[:5],
    }


def collect_news(sources=SOURCES, n_per_feed=N_PER_FEED, scraper=scraper):
    """Parcourt tous les flux et renvoie un DataFrame d'articles."""
    rows, seen = [], set()
    for media, feeds in sources.items():
        for category, feed_url in feeds.items():
            try:
                entries = read_feed(feed_url, scraper)
            except requests.RequestException as error:
                print(f"Flux ignoré ({media} / {category}) : {error}")
                continue

            count = 0
            for entry in entries:
                if count >= n_per_feed:
                    break
                if entry["url"] in seen:  # un article peut être dans plusieurs flux
                    continue
                seen.add(entry["url"])
                try:
                    rows.append(scrape_article(entry, category, media, scraper))
                    count += 1
                except requests.RequestException as error:
                    print(f"Article ignoré : {entry['url']} ({error})")
                time.sleep(PAUSE)
            print(f"{media:<11} | {category:<10} : {count} article(s)")
    return pd.DataFrame(rows)


In [25]:
df = collect_news()
print(f"\n{len(df)} articles récupérés")
df[["media", "category", "article_title", "domain", "text_source", "text_length"]]


franceinfo  | Politique  : 3 article(s)


franceinfo  | Économie   : 3 article(s)


franceinfo  | Sport      : 3 article(s)


franceinfo  | Culture    : 3 article(s)


franceinfo  | Sciences   : 3 article(s)


20 Minutes  | Économie   : 3 article(s)


20 Minutes  | Sport      : 3 article(s)


20 Minutes  | Culture    : 3 article(s)


20 Minutes  | Sciences   : 3 article(s)

27 articles récupérés


,media,category,article_title,domain,text_source,text_length
0,franceinfo,Politique,"Age de départ, part de capitalisation, possibl...",franceinfo.fr,article,4898
1,franceinfo,Politique,Y a-t-il trop de fonctionnaires en France ?,franceinfo.fr,résumé RSS,260
2,franceinfo,Politique,"""Des accusations fallacieuses, des méthodes tr...",franceinfo.fr,résumé RSS,281
3,franceinfo,Économie,Comment bien se servir de l'IA pour préparer s...,franceinfo.fr,article,311
4,franceinfo,Économie,Pompe à chaleur : les premières offres de leas...,franceinfo.fr,article,2289
5,franceinfo,Économie,25 000 offres d'emploi supprimées sur France T...,franceinfo.fr,article,2417
6,franceinfo,Sport,Arthur Fils se défait de Rafael Jodar au Maste...,franceinfo.fr,article,2207
7,franceinfo,Sport,"""Des dérives qui ont introduit le poison de la...",franceinfo.fr,article,5814
8,franceinfo,Sport,"""J'en suis arrivé à un point où tu envisages l...",franceinfo.fr,article,2486
9,franceinfo,Culture,Le chanteur Aldebert sort un best of et prépar...,franceinfo.fr,article,3055


Exemple d'un article complet :

In [26]:
article = df.iloc[0]
for col in ["media", "category", "article_title", "page_title", "url", "domain", "published", "text_source"]:
    print(f"{col:<14}: {article[col]}")
print("images        :", article["images"])
print("\nTexte :\n", article["text"][:1500], "...")


media         : franceinfo
category      : Politique
article_title : Age de départ, part de capitalisation, possible référendum… Que propose le Rassemblement national pour réformer les retraites ?
page_title    : Age de départ, part de capitalisation, possible référendum… Que propose le Rassemblement national pour réformer les retraites ? – franceinfo
url           : https://www.franceinfo.fr/economie/retraite/reforme-des-retraites/age-de-depart-part-de-capitalisation-referendum-que-propose-le-rassemblement-national-pour-reformer-les-retraites_8286419.html
domain        : franceinfo.fr
published     : Tue, 17 Nov 2026 14:03:56 +0100
text_source   : article
images        : ['https://www.franceinfo.fr/pictures/kju0IMgnzFZf7WgWf35Rq8Gnhy4/0x27:4963x2818/1500x843/2026/11/17/6afc369759dd4448045503.jpg', 'https://www.franceinfo.fr/pictures/yeiJ5cikrD86IE8y89XRMnTnQUQ/0x27:4963x2818/432x243/filters:format(jpg)/2026/11/17/6afc369759dd4448045503.jpg', 'https://www.franceinfo.fr/pictures/0Xh27lZ

Nombre d'articles par média et par catégorie, puis sauvegarde dans un fichier CSV :

In [27]:
print(df.groupby(["media", "category"]).size())

df.to_csv("articles_news.csv", index=False)
print("\nSauvegardé dans 1Introduction/articles_news.csv")


media       category 
20 Minutes  Culture      3
            Sciences     3
            Sport        3
            Économie     3
franceinfo  Culture      3
            Politique    3
            Sciences     3
            Sport        3
            Économie     3
dtype: int64

Sauvegardé dans 1Introduction/articles_news.csv
